<a href="https://colab.research.google.com/github/Yousef-Shihade/information-retrieval-course-tasks/blob/main/Task_02_BM25_Ranking/BM25_Ranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# option 2 – Okapi BM25 with k1 = 1.6, b = 0.75
# YOUSEF SHIHADE

import math
import pandas as pd

# 10 terms
vocab = [
    "information", "retrieval", "course", "python", "search",
    "index", "document", "query", "ranking", "model"
]

#  input : 6 documents, counts of each term
documents_tf = {
    "D1":  {"information": 2, "retrieval": 1, "course": 1, "python": 0, "search": 1,
            "index": 0, "document": 1, "query": 0, "ranking": 0, "model": 0},
    "D2":  {"information": 1, "retrieval": 2, "course": 1, "python": 1, "search": 1,
            "index": 1, "document": 0, "query": 0, "ranking": 0, "model": 0},
    "D3":  {"information": 1, "retrieval": 1, "course": 0, "python": 1, "search": 2,
            "index": 1, "document": 1, "query": 0, "ranking": 1, "model": 0},
    "D4":  {"information": 1, "retrieval": 0, "course": 1, "python": 2, "search": 1,
            "index": 0, "document": 1, "query": 1, "ranking": 0, "model": 0},
    "D5":  {"information": 2, "retrieval": 1, "course": 0, "python": 1, "search": 1,
            "index": 1, "document": 0, "query": 1, "ranking": 1, "model": 0},
    "D6":  {"information": 1, "retrieval": 1, "course": 1, "python": 0, "search": 0,
            "index": 1, "document": 1, "query": 1, "ranking": 0, "model": 1},
}

# DataFrame: rows = documents, columns = terms, cells = counts
df_counts = (
    pd.DataFrame.from_dict(documents_tf, orient="index")[vocab]
    .fillna(0)
    .astype(int)
)

print("Input: term frequencies (counts) ")
print(df_counts)
print()

# query: 3–4 terms from the vocabulary
query_terms = ["information", "retrieval", "python", "ranking"]  # 4 terms
query_terms = [t for t in query_terms if t in vocab]

print(" Query terms : ", query_terms)
print()

#  BM25 parameters
k1 = 1.6
b = 0.75

#  BM25 calculations
N = df_counts.shape[0]              # num of doc
doc_len = df_counts.sum(axis=1)     # len of each doc (terms)
avgdl = doc_len.mean()              # average doc len

# docu frequency
df = (df_counts > 0).sum(axis=0)

# IDF for each term -> Okapi BM25
idf = {}
for term in vocab:
    df_t = df[term]
    idf[term] = math.log((N - df_t + 0.5) / (df_t + 0.5) + 1)

# DataFrame for BM25 score of each term in each doc
bm25_terms = pd.DataFrame(0.0, index=df_counts.index, columns=vocab)

for term in query_terms:
    for doc_id in df_counts.index:
        f = df_counts.loc[doc_id, term]   # term frequency in this doc
        if f == 0:
            continue
        dl = doc_len[doc_id]
        numerator = f * (k1 + 1)
        denominator = f + k1 * (1 - b + b * dl / avgdl)
        bm25_terms.loc[doc_id, term] = idf[term] * numerator / denominator

# total BM25 score per doc = sum of BM25 of the query terms
doc_scores = bm25_terms[query_terms].sum(axis=1)

# output DataFrame
# first column: total score, then BM25 of each term (same layout as input)
df_output = pd.concat([doc_scores.rename("score"), bm25_terms], axis=1)

print(" Output : BM25 scores (per document & per term) ")
print(df_output)
print()

#  show ranking of doc by score
print("Documents ranked by BM25 score (highest first) ")
print(df_output.sort_values("score", ascending=False)[["score"]])


Input: term frequencies (counts) 
    information  retrieval  course  python  search  index  document  query  \
D1            2          1       1       0       1      0         1      0   
D2            1          2       1       1       1      1         0      0   
D3            1          1       0       1       2      1         1      0   
D4            1          0       1       2       1      0         1      1   
D5            2          1       0       1       1      1         0      1   
D6            1          1       1       0       0      1         1      1   

    ranking  model  
D1        0      0  
D2        0      0  
D3        1      0  
D4        0      0  
D5        1      0  
D6        0      1  

 Query terms :  ['information', 'retrieval', 'python', 'ranking']

 Output : BM25 scores (per document & per term) 
       score  information  retrieval  course    python  search  index  \
D1  0.373940     0.113187   0.260754     0.0  0.000000     0.0    0.0   
D2  0.872